# FDS Challenge: My Notebook

## 0. Configuration

In [ ]:
import json
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


DEBUG = True


# --- PATHS ---
TRAIN_FILE_PATH        = "../dataset/train.jsonl"
TEST_FILE_PATH         = "../dataset/test.jsonl"
TYPE_CHART_PATH        = "../dataset/type_chart_gen2.csv"
TYPE_MAP_PATH          = "type_map.json"
SAMPLE_SUBMISSION_PATH = "../dataset/sample_submission.csv"


# --- FEATURES ---
BAD_STATUSES = {
    # fnt is not actual a status
    "fnt": -1,
    "slp": 3,
    "frz": 3,
    "tox": 2,
    "psn": 1,
    "par": 2,
    "brn": 2
}
MULTS_WEIGHTS = {
    0:    -3,
    0.25: -2,
    0.5:  -1,
    1:     0,
    2:     1,
    4:     2
}

EFFECT_WEIGHTS = {
    # --- HIGH-FREQUENCY POSITIVE (Logged on USER) ---
    "substitute":   1.5,  # 1,348 battles (56.1% WR): Blocks status, absorbs hits, huge momentum
    "destinybond":  1.2,  # 247 battles (56.3% WR): Forces 1-for-1 trade or forces opponent out
    "perish":       0.3,  # 675 battles (51.4% WR): Logged on both, slight edge to Perish-trap initiator
    # --- LOW-FREQUENCY POSITIVE (Shrunk to prevent small-sample overfitting) ---
    "safeguard":    0.6,  # 3 battles (100% WR): Team status immunity for 5 turns
    "bide":         0.4,  # 8 battles (87.5% WR): Logged on user
    "typechange":   0.3,  # 5 battles (60.0% WR): Porygon2 Conversion2 defensive typing
    "beatup":       0.3,  # 3 battles (66.7% WR): Logged on user attacking
    "magnitude":    0.3,  # 9 battles (66.7% WR): Logged on user attacking
    "lockon":       0.2,  # 1 battle (100% WR): Shrunk due to N=1
    # --- NEUTRAL ---
    "noeffect":     0.0,  # Default empty state
    # --- HIGH-FREQUENCY NEGATIVE (Logged on TARGET/VICTIM or Desperation Stall) ---
    "pursuit":      -1.5,  # 76 battles (35.5% WR): Victim caught switching out for 2x damage
    "encore":       -1.2,  # 152 battles (45.4% WR): Victim locked into one move, free setup for opp
    "clamp":        -1.0,  # 150 battles (45.3% WR): Victim trapped and taking residual damage
    "nightmare":    -1.0,  # 96 battles (46.9% WR): Victim losing 25% HP/turn while asleep
    "trapped":      -0.9,  # 1,206 battles (48.1% WR): Victim trapped by Mean Look / Spider Web
    "confusion":    -0.8,  # 2,001 battles (47.8% WR): Victim has 50% chance to hit itself
    "leechseed":    -0.8,  # 893 battles: Victim loses 12.5% HP/turn and heals opponent
    "protect":      -0.6,  # 881 battles (44.4% WR): Empirically indicates stalling when behind/losing
    "attract":      -0.5,  # 99 battles (50.5% WR): Victim immobilized 50% of turns
    # --- LOW-FREQUENCY NEGATIVE (Traps, Ailments & Desperation/Gimmick Markers) ---
    "curse":        -0.8,  # 19 battles: Verified TARGET (Ghost Curse), loses 25% HP/turn
    "whirlpool":    -0.4,  # 40 battles: Partial trapping damage on victim
    "wrap":         -0.4,  # 19 battles: Partial trapping damage on victim
    "firespin":     -0.4,  # 15 battles: Partial trapping damage on victim
    "bind":         -0.4,  # 5 battles: Partial trapping damage on victim
    "disable":      -0.4,  # 8 battles (50.0% WR): Disables victim's move
    "spite":        -0.3,  # 2 battles: Drains victim's PP
    "foresight":    -0.2,  # 1 battle: Logged on target (exposes Ghost to Rapid Spin/Fighting)
    "endure":       -0.5,  # 34 battles (47.1% WR): Desperation survival at 1 HP
    "focusenergy":  -0.4,  # 13 battles (46.2% WR): Suboptimal gimmick move in Gen 2 OU
    "mimic":        -0.6,  # 11 battles (36.4% WR): Gimmick move, strong negative empirical signal
    "futuresight":  -0.7,  # 7 battles (14.3% WR): Suboptimal move choice in Gen 2 OU
    "mindreader":   -0.5,  # 4 battles (25.0% WR): Gimmick move
    "focusband":    -0.8,  # 11 battles (18.2% WR): Suboptimal item + means holder took lethal hit
    "quickclaw":    -0.6,  # 7 battles (28.6% WR): Suboptimal item (no Leftovers recovery)
    "mysteryberry": -0.3,  # 1 battle (0.0% WR): Shrunk due to N=1
}

# --- TRAINING ---
"""
"logistic_regression":
A simple linear model that weights each feature and outputs a win probability.
It is fast and easy to interpret, but it only captures straightforward relationships.
Features are standardized first.

"random_forest":
Trains 150 decision trees on different random subsets of the data and averages their votes.
It handles non-linear patterns well and is robust to noise and overfitting.

"lightgbm":
A gradient boosting model that builds trees one after another, each correcting the errors of the previous ones.
It is usually the most accurate on tabular data.

"ensemble":
A soft-voting combination of the three models above, averaging their probabilities with weights.
It tends to be the most stable, but it is the slowest because it trains three models per fold.
"""
MODEL_TYPE = "ensemble"
RANDOM_STATE = 42
K_FOLDS = 5
N_REPEATS = 3


# --- BATTLE EXAMPLE ---
def print_battle(data, battle_idx=0):
    """
    Prints the whole battle at index `battle_idx`
    """

    battle = data[battle_idx]
    output_path = f"battle_{battle_idx}.json"
    with open(output_path, "w") as f:
        json.dump(battle, f, indent=4)

    print(f"SUCCESS: battle saved to {output_path}.")

## 1. Loading the Data

In [ ]:
# --- LOAD DATA ---
def load_data(file_path):
    """
    Loads the whole `file_path` and stores it as a list of battles
    """

    print(f"Loading data from '{file_path}'...")
    data = []
    try:
        with open(file_path, "r") as f:
            for line in f:
                data.append(json.loads(line))
        print(f"SUCCESS: loaded {len(data)} battles.")
        if DEBUG:
            print_battle(data)
    except FileNotFoundError:
        print(f"ERROR: Could not find the file at '{file_path}'.")

    return data


# Loading data
train_data = load_data(TRAIN_FILE_PATH)
test_data  = load_data(TEST_FILE_PATH)

### Types Charts

In [ ]:
# -- TYPE CHART ---
def load_type_chart(file_path):
    try:
        return pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"ERROR: Could not find the file at '{file_path}'.")


# --- POKEMON TYPE MAP ---
def create_type_map(train_data, test_data, file_path=TYPE_MAP_PATH):
    """
    Build the type map, save it as JSON, and return it
    """

    type_map = {}
    for data in (train_data, test_data):
        for battle in data:
            for pokemon in battle["p1_team_details"]:
                type_map[pokemon["name"]] = pokemon["types"]

            lead = battle["p2_lead_details"]
            type_map[lead["name"]] = lead["types"]

    with open(file_path, "w") as f:
        json.dump(type_map, f, indent=4)
    return type_map


def get_multiplier(move_type, pkm_type1, pkm_type2):
    """
    Return the move's combined effectiveness against the pokemon's types
    """

    m1 = type_chart.get((move_type, pkm_type1), 1.0) if pkm_type1 else 1.0
    m2 = type_chart.get((move_type, pkm_type2), 1.0) if pkm_type2 else 1.0
    return m1 * m2


# Getting type charts
type_chart = load_type_chart(TYPE_CHART_PATH)
df_indexed = type_chart.set_index(["attacking_type", "defending_type"])
type_chart = df_indexed["multiplier"].to_dict()
type_map = create_type_map(train_data, test_data)

## 2. Extract Features (both Train and Test Features)

In [ ]:
# --- BASE STATS ---
def feature_base_stats(battle, features):
    """
    Add base pokemon statistics of:
    - player 1 full team
    - player 2 lead pokemon
    - player 1 mean pokemon stats - player 2 lead pokemon stats
    """

    p1_team = battle["p1_team_details"]
    features["p1_mean_hp"]  = np.mean([p["base_hp"]  for p in p1_team])
    features["p1_mean_atk"] = np.mean([p["base_atk"] for p in p1_team])
    features["p1_mean_def"] = np.mean([p["base_def"] for p in p1_team])
    features["p1_mean_spa"] = np.mean([p["base_spa"] for p in p1_team])
    features["p1_mean_spd"] = np.mean([p["base_spd"] for p in p1_team])
    features["p1_mean_spe"] = np.mean([p["base_spe"] for p in p1_team])
    features["p1_mean_bst"] = features["p1_mean_hp"] + features["p1_mean_atk"] + features["p1_mean_def"] + features["p1_mean_spa"] + features["p1_mean_spd"] + features["p1_mean_spe"]

    p2_lead = battle["p2_lead_details"]
    features["p2_lead_hp"]  = p2_lead["base_hp"]
    features["p2_lead_atk"] = p2_lead["base_atk"]
    features["p2_lead_def"] = p2_lead["base_def"]
    features["p2_lead_spa"] = p2_lead["base_spa"]
    features["p2_lead_spd"] = p2_lead["base_spd"]
    features["p2_lead_spe"] = p2_lead["base_spe"]
    features["p2_lead_bst"] = features["p2_lead_hp"] + features["p2_lead_atk"] + features["p2_lead_def"] + features["p2_lead_spa"] + features["p2_lead_spd"] + features["p2_lead_spe"]

    features["diff_mean_hp"]  = features["p1_mean_hp"]  - features["p2_lead_hp"]
    features["diff_mean_atk"] = features["p1_mean_atk"] - features["p2_lead_atk"]
    features["diff_mean_def"] = features["p1_mean_def"] - features["p2_lead_def"]
    features["diff_mean_spa"] = features["p1_mean_spa"] - features["p2_lead_spa"]
    features["diff_mean_spd"] = features["p1_mean_spd"] - features["p2_lead_spd"]
    features["diff_mean_spe"] = features["p1_mean_spe"] - features["p2_lead_spe"]
    features["diff_mean_bst"] = features["p1_mean_bst"] - features["p2_lead_bst"]


# --- HP LOSS ---
def get_hp_pct_lost(index, turn, player, team):
    """
    Track HP loss from one turn for each observed pokemon:
    - remember its latest HP across switches, starting from full HP
    - accumulate HP loss in its three ten-turn intervals
    Healing reduces HP loss.
    """

    name = turn[f"p{player}_pokemon_state"]["name"]
    hp = turn[f"p{player}_pokemon_state"]["hp_pct"]
    if name not in team:
        team[name] = {"last_hp": 1.0, "losses": [0.0, 0.0, 0.0]}

    pokemon = team[name]
    pokemon["losses"][index // 10] += pokemon["last_hp"] - hp
    pokemon["last_hp"] = hp


def feature_hp_lost(p1_hp_map, p2_hp_map, features):
    """
    Add mean HP loss for both players after all observed turns:
    - sum each pokemon's loss in turns 0-10, 10-20, and 20-30
    - divide each interval total by the final number of observed team pokemon
    Partial intervals (eg. battle ended before 30 turns) are valid too.
    Unobserved intervals remain zero.
    """

    features["p1_hp_pct_lost_0_10"]  = sum(p["losses"][0] for p in p1_hp_map.values()) / len(p1_hp_map)
    features["p1_hp_pct_lost_10_20"] = sum(p["losses"][1] for p in p1_hp_map.values()) / len(p1_hp_map)
    features["p1_hp_pct_lost_20_30"] = sum(p["losses"][2] for p in p1_hp_map.values()) / len(p1_hp_map)

    features["p2_hp_pct_lost_0_10"]  = sum(p["losses"][0] for p in p2_hp_map.values()) / len(p2_hp_map)
    features["p2_hp_pct_lost_10_20"] = sum(p["losses"][1] for p in p2_hp_map.values()) / len(p2_hp_map)
    features["p2_hp_pct_lost_20_30"] = sum(p["losses"][2] for p in p2_hp_map.values()) / len(p2_hp_map)


# --- BAD STATUSES + FAINT ---
def get_statuses(turn, p1_status, p2_status):
    """
    Get:
    - player 1 active pokemon status
    - player 2 active pokemon status
    """

    pkm1_status = turn["p1_pokemon_state"]["status"]
    if pkm1_status != "nostatus":
        p1_status[pkm1_status] += 1

    pkm2_status = turn["p2_pokemon_state"]["status"]
    if pkm2_status != "nostatus":
        p2_status[pkm2_status] += 1


def feature_statuses(p1_status_map, p2_status_map, features):
    """
    Add weighted features using each pokemon's latest observed status:
    - each bad status count multiplied by its weight, for both players
    - each player's sum of weighted status features
    - player 1 weighted sum - player 2 weighted sum
    """

    features["p1_fnt"] = p1_status_map["fnt"]
    features["p1_slp"] = p1_status_map["slp"]
    features["p1_frz"] = p1_status_map["frz"]
    features["p1_tox"] = p1_status_map["tox"]
    features["p1_psn"] = p1_status_map["psn"]
    features["p1_par"] = p1_status_map["par"]
    features["p1_brn"] = p1_status_map["brn"]
    # fnt is not actual a status
    features["p1_statuses_wmean"] = sum([p1_status_map[s] * BAD_STATUSES[s] for s in BAD_STATUSES.keys() if s != "fnt"]) / (len(BAD_STATUSES) - 1)

    features["p2_fnt"] = p2_status_map["fnt"]
    features["p2_slp"] = p2_status_map["slp"]
    features["p2_frz"] = p2_status_map["frz"]
    features["p2_tox"] = p2_status_map["tox"]
    features["p2_psn"] = p2_status_map["psn"]
    features["p2_par"] = p2_status_map["par"]
    features["p2_brn"] = p2_status_map["brn"]
    # fnt is not actual a status
    features["p2_statuses_wmean"] = sum([p2_status_map[s] * BAD_STATUSES[s] for s in BAD_STATUSES.keys() if s != "fnt"]) / (len(BAD_STATUSES) - 1)    


# --- ACTIVE HP ---
def feature_active_hp(turn, features):
    """
    Add active pokemon HP at the end of the observed timeline:
    - player 1 active pokemon HP fraction
    - player 2 active pokemon HP fraction
    """

    features["p1_active_hp"] = turn["p1_pokemon_state"]["hp_pct"]
    features["p2_active_hp"] = turn["p2_pokemon_state"]["hp_pct"]


# --- BOOSTS ---
def feature_boosts(turn, features):
    """
    Add active pokemon stat boosts at the end of the observed timeline:
    - player 1 active pokemon boosts
    - player 2 active pokemon boosts
    """

    features["p1_boost_atk"] = turn["p1_pokemon_state"]["boosts"]["atk"]
    features["p1_boost_def"] = turn["p1_pokemon_state"]["boosts"]["def"]
    features["p1_boost_spa"] = turn["p1_pokemon_state"]["boosts"]["spa"]
    features["p1_boost_spd"] = turn["p1_pokemon_state"]["boosts"]["spd"]
    features["p1_boost_spe"] = turn["p1_pokemon_state"]["boosts"]["spe"]

    features["p2_boost_atk"] = turn["p2_pokemon_state"]["boosts"]["atk"]
    features["p2_boost_def"] = turn["p2_pokemon_state"]["boosts"]["def"]
    features["p2_boost_spa"] = turn["p2_pokemon_state"]["boosts"]["spa"]
    features["p2_boost_spd"] = turn["p2_pokemon_state"]["boosts"]["spd"]
    features["p2_boost_spe"] = turn["p2_pokemon_state"]["boosts"]["spe"]


# --- TYPE EFFECTIVENESS ---
def get_mults(turn, p1_mults, p2_mults):
    """
    Update both players' type effectiveness multiplier counts for one turn
    """
    
    mult1, mult2 = 1, 1

    mv1 = turn["p1_move_details"]
    if mv1 != None:
        mv1_category = mv1["category"]
        if mv1_category != "status":
            mv1_type = mv1["type"]
            pkm2_name = turn["p2_pokemon_state"]["name"]
            pkm2_types = type_map.get(pkm2_name, ["notype", "notype"])
            mult1 = get_multiplier(mv1_type, pkm2_types[0], pkm2_types[1])
        p1_mults[mult1] += 1

    mv2 = turn["p2_move_details"]
    if mv2 != None:
        mv2_category = mv2["category"]
        if mv2_category != "status":
            mv2_type = mv2["type"]
            pkm1_name = turn["p1_pokemon_state"]["name"]
            # pkm1 types are always known
            pkm1_types = type_map[pkm1_name]
            mult2 = get_multiplier(mv2_type, pkm1_types[0], pkm1_types[1])

    p2_mults[mult2] += 1


def feature_mults(p1_mults, p2_mults, features):
    """
    Add multiplier counts and weighted scores for both players
    """

    features["p1_0_mult"]    = p1_mults[0]
    features["p1_0.25_mult"] = p1_mults[0.25]
    features["p1_0.5_mult"]  = p1_mults[0.5]
    features["p1_1_mult"]    = p1_mults[1]
    features["p1_2_mult"]    = p1_mults[2]
    features["p1_4_mult"]    = p1_mults[4]
    features["p1_mult_wmean"] = sum([p1_mults[x] * MULTS_WEIGHTS[x] for x in MULTS_WEIGHTS.keys()]) / len(MULTS_WEIGHTS)

    features["p2_0_mult"]    = p2_mults[0]
    features["p2_0.25_mult"] = p2_mults[0.25]
    features["p2_0.5_mult"]  = p2_mults[0.5]
    features["p2_1_mult"]    = p2_mults[1]
    features["p2_2_mult"]    = p2_mults[2]
    features["p2_4_mult"]    = p2_mults[4]
    features["p2_mult_wmean"] = sum([p2_mults[x] * MULTS_WEIGHTS[x] for x in MULTS_WEIGHTS.keys()]) / len(MULTS_WEIGHTS)


# --- EFFECTS FEATURES ---
def get_effects(turn, p1_effects, p2_effects):
    """
    Accumulate positive and negative effect weights from one turn for both players
    """

    for eff in turn["p1_pokemon_state"]["effects"]:
        w = EFFECT_WEIGHTS.get(eff, 0.0)
        if w > 0:
            p1_effects["pos"] += w
        elif w < 0:
            p1_effects["neg"] += w

    for eff in turn["p2_pokemon_state"]["effects"]:
        w = EFFECT_WEIGHTS.get(eff, 0.0)
        if w > 0:
            p2_effects["pos"] += w
        elif w < 0:
            p2_effects["neg"] += w

def feature_effects(p1_effects, p2_effects, last_turn, n_turns, features):
    """
    Add weighted effect scores after all observed turns and at the final turn:
    - mean positive, negative, and net effect weights for P1 and P2
    - net weighted effect difference (P1 - P2) across the battle
    - active effect scores and difference at the end of the timeline
    """

    features["p1_eff_pos_mean"] = p1_effects["pos"] / n_turns
    features["p1_eff_neg_mean"] = p1_effects["neg"] / n_turns
    features["p2_eff_pos_mean"] = p2_effects["pos"] / n_turns
    features["p2_eff_neg_mean"] = p2_effects["neg"] / n_turns

    features["p1_eff_net_mean"] = features["p1_eff_pos_mean"] + features["p1_eff_neg_mean"]
    features["p2_eff_net_mean"] = features["p2_eff_pos_mean"] + features["p2_eff_neg_mean"]
    features["diff_eff_net_mean"] = features["p1_eff_net_mean"] - features["p2_eff_net_mean"]

    p1_last_eff = sum(EFFECT_WEIGHTS.get(e, 0.0) for e in last_turn["p1_pokemon_state"]["effects"])
    p2_last_eff = sum(EFFECT_WEIGHTS.get(e, 0.0) for e in last_turn["p2_pokemon_state"]["effects"])
    features["p1_last_eff_score"] = p1_last_eff
    features["p2_last_eff_score"] = p2_last_eff
    features["diff_last_eff_score"] = p1_last_eff - p2_last_eff

In [ ]:
def extract_features(data):
    """
    Extract the features:
    1. base stats
    2. HP loss in ten-turn intervals
    3. bad statuses
    4. active HP at the end
    5. individual stat boosts at the end
    6. Type effectiveness multiplier counts
    7. Active and cumulative battle effects
    """

    print("Extracting features...")
    feature_list = []

    for battle in data:
        features = {}

        # 1. Base stats
        feature_base_stats(battle, features)

        p1_hp_map, p2_hp_map = {}, {}
        p1_status = {key: 0 for key in BAD_STATUSES}
        p2_status = {key: 0 for key in BAD_STATUSES}
        p1_mults = {key: 0 for key in MULTS_WEIGHTS}
        p2_mults = {key: 0 for key in MULTS_WEIGHTS}
        p1_effects = {"pos": 0.0, "neg": 0.0}
        p2_effects = {"pos": 0.0, "neg": 0.0}

        timeline = battle["battle_timeline"]
        for index, turn in enumerate(battle["battle_timeline"]):
            get_statuses(turn, p1_status, p2_status)
            get_hp_pct_lost(index, turn, 1, p1_hp_map)
            get_hp_pct_lost(index, turn, 2, p2_hp_map)
            get_mults(turn, p1_mults, p2_mults)
            get_effects(turn, p1_effects, p2_effects)

        last_turn = timeline[-1]

        # 2. HP loss in ten-turn intervals
        feature_hp_lost(p1_hp_map, p2_hp_map, features)
        # 3. Bad statuses
        feature_statuses(p1_status, p2_status, features)
        # 4. Active HP at the end
        feature_active_hp(last_turn, features)
        # 5. Individual stat boosts at the end
        feature_boosts(last_turn, features)
        # 6. Type effectiveness multiplier counts
        feature_mults(p1_mults, p2_mults, features)
        # 7. Active and cumulative battle effects
        feature_effects(p1_effects, p2_effects, last_turn, len(timeline), features)

        # Keep the identifier and target for the training and submission cells
        if "battle_id" in battle:
            features["battle_id"] = battle["battle_id"]
        if "player_won" in battle:
            features["player_won"] = int(battle["player_won"])

        feature_list.append(features)

    print(f"SUCCESS: extracted {len(feature_list[0])} features.")
    return pd.DataFrame(feature_list).fillna(0)


# Extracting training and test features
train_df = extract_features(train_data)
print(f"Training features: {train_df.shape}.")
display(train_df.head())

test_df = extract_features(test_data)
print(f"Test features: {test_df.shape}.")
# display(test_df.head())

## 3. Training

### Perfomance Metrics

In [ ]:
def compute_metrics(y_true, y_pred, y_prob):
    return {
        # (TP + TN) / total
        "accuracy": accuracy_score(y_true, y_pred),
        # TP / (TP + FP)
        "precision": precision_score(y_true, y_pred, zero_division=0),
        # TP / (TP + FN)
        "recall": recall_score(y_true, y_pred, zero_division=0),
        # 2 * (precision * recall) / (precision + recall) =
        # 2 * TP / (2 * TP + FP + FN)
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

In [ ]:
def make_lgbm(random_state):
    return LGBMClassifier(
        n_estimators=150,
        max_depth=5,
        learning_rate=0.04,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        verbose=-1,
        random_state=random_state,
    )


def make_rf(random_state):
    return RandomForestClassifier(
        n_estimators=150,
        max_depth=8,
        min_samples_split=5,
        random_state=random_state,
        n_jobs=-1,
    )


def make_lr(random_state):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=1.0, random_state=random_state),
    )


# --- BUILD MODEL ---
def build_model(model_type=MODEL_TYPE, random_state=RANDOM_STATE):
    """
    Build an instance of `model_type` with seed `random_state`
    """

    match model_type:
        case "logistic_regression":
            return make_lr(random_state)

        case "random_forest":
            return make_rf(random_state)

        case "lightgbm":
            return make_lgbm(random_state)

        case "ensemble":
            return VotingClassifier(
                estimators=[
                    ("lgb", make_lgbm(random_state)),
                    ("rf", make_rf(random_state)),
                    ("lr", make_lr(random_state)),
                ],
                voting="soft",
                weights=[0.5, 0.3, 0.2],
            )

        case _:
            raise ValueError(f"ERROR: Unavailable model: {model_type}.")


# --- TRAINING ---
def train_and_evaluate_cv(X, y, X_test, model_type=MODEL_TYPE, k=K_FOLDS, n_repeats=N_REPEATS, random_state=RANDOM_STATE):
    """
    Repeated Stratified K-Fold Cross-Validation:
    1. Computes metrics on each of the k * n_repeats validation folds.
    2. Computes Out-Of-Fold (OOF) probabilities, averaged across repeats.
    3. Averages the test-set probabilities over all k * n_repeats models.
    """
    rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=random_state)
    n_total = k * n_repeats

    oof_probs = np.zeros(len(X))
    test_probs = np.zeros(len(X_test))
    fold_metrics_list = []

    print(f"\n{'='*60}")
    print(f"Avvio {n_repeats}x Stratified {k}-Fold CV con modello: {model_type.upper()}")
    print(f"{'='*60}")

    for i, (train_idx, val_idx) in enumerate(rskf.split(X, y)):
        repeat, fold = divmod(i, k)
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = build_model(model_type, random_state=random_state)
        model.fit(X_train, y_train)

        # Predizione sul fold di validazione
        val_prob = model.predict_proba(X_val)[:, 1]
        val_pred = (val_prob >= 0.5).astype(int)
        oof_probs[val_idx] += val_prob / n_repeats

        metrics = compute_metrics(y_val, val_pred, val_prob)
        fold_metrics_list.append(metrics)

        print(f"Repeat {repeat+1}/{n_repeats} | Fold {fold+1}/{k} -> " f"Accuracy: {metrics['accuracy']:.4f} | F1: {metrics['f1']:.4f} | " f"ROC-AUC: {metrics['roc_auc']:.4f}")

        # Accumula predizione sul test set
        test_probs += model.predict_proba(X_test)[:, 1] / n_total

    # Statistiche aggregate sui K fold
    mean_acc = np.mean([m["accuracy"] for m in fold_metrics_list])
    std_acc = np.std([m["accuracy"] for m in fold_metrics_list])
    mean_f1 = np.mean([m["f1"] for m in fold_metrics_list])
    mean_auc = np.mean([m["roc_auc"] for m in fold_metrics_list])

    # Performance Out-Of-Fold complessiva
    oof_pred = (oof_probs >= 0.5).astype(int)
    overall_oof_acc = accuracy_score(y, oof_pred)
    overall_oof_auc = roc_auc_score(y, oof_probs)

    print(f"\n{'-'*60}")
    print(f"RISULTATI CV AGGREGATI ({n_repeats}x{k} FOLDS = {n_total} valutazioni):")
    print(f"Accuracy Media : {mean_acc:.4f} (+/- {std_acc:.4f})")
    print(f"F1 Media       : {mean_f1:.4f}")
    print(f"ROC-AUC Media  : {mean_auc:.4f}")
    print(f"OOF Accuracy   : {overall_oof_acc:.4f}")
    print(f"OOF ROC-AUC    : {overall_oof_auc:.4f}")
    print(f"{'-'*60}\n")

    test_predictions = (test_probs >= 0.5).astype(int)
    return test_predictions, test_probs


# Selezione delle feature escludendo ID e target
feature_cols = [c for c in train_df.columns if c not in ["battle_id", "player_won"]]
print(f"\nNumero di feature utilizzate: {len(feature_cols)}")
print(f"Elenco feature: {feature_cols}")

X_train = train_df[feature_cols]
y_train = train_df["player_won"]
X_test = test_df[feature_cols]

# 3. Addestramento, Validazione K-Fold e Predizione Ensembled
test_predictions, test_probabilities = train_and_evaluate_cv(
    X=X_train,
    y=y_train,
    X_test=X_test,
    model_type=MODEL_TYPE,
    k=K_FOLDS,
    random_state=RANDOM_STATE,
)

## 4. Submission File

In [ ]:
def predict_and_save(predictions, test_battle_ids, output_path="submission.csv"):
    """
    Formatta e salva il file di sottomissione secondo il template richiesto.
    """
    print(f"Salvataggio delle predizioni in '{output_path}'...")
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    expected_columns = ["battle_id", "player_won"]
    if list(sample_submission.columns) != expected_columns:
        raise ValueError(f"Expected submission columns {expected_columns}, got {list(sample_submission.columns)}")

    predictions_by_id = pd.Series(predictions, index=test_battle_ids)
    submission_df = sample_submission.copy()
    submission_df["player_won"] = submission_df["battle_id"].map(predictions_by_id)

    if submission_df["player_won"].isna().any():
        raise ValueError("Alcuni battle_id nella sample submission non hanno una predizione corrispondente.")

    submission_df["player_won"] = submission_df["player_won"].astype(int)
    submission_df.to_csv(output_path, index=False)

    print(f"File '{output_path}' creato con successo!")
    print(submission_df.head(10))
    print(f"Distribuzione predizioni (0 vs 1): {dict(submission_df['player_won'].value_counts())}")


# 4. Creazione del file di sottomissione
predict_and_save(test_predictions, test_battle_ids=test_df["battle_id"], output_path="submission.csv")

## 5. Model evaluation


In [ ]:
import sys
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold

# ==============================================================================
# 5. BENCHMARK E CONFRONTO MULTI-MODELLO
# ==============================================================================
# Elenco dei modelli definiti a inizio notebook
MODELS_TO_BENCHMARK = [
    "logistic_regression",
    "random_forest",
    "lightgbm",
    "ensemble",
]

BENCHMARK_OUTPUT_FILE = "all_models_output.txt"
HISTOGRAM_OUTPUT_FILE = "models_comparison_histogram.png"


class TeeOutput:
    """
    Duplica l'output standard (stdout) sia sulla console/notebook che su file.
    Garantisce che tutti gli output di esecuzione vengano salvati in un unico file.
    """

    def __init__(self, file_path, mode="w", encoding="utf-8"):
        self.file_path = file_path
        self.mode = mode
        self.encoding = encoding
        self.file = None
        self.stdout = None

    def __enter__(self):
        self.file = open(self.file_path, self.mode, encoding=self.encoding)
        self.stdout = sys.stdout
        sys.stdout = self
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout = self.stdout
        if self.file and not self.file.closed:
            self.file.close()

    def write(self, data):
        self.stdout.write(data)
        self.file.write(data)

    def flush(self):
        self.stdout.flush()
        self.file.flush()


def evaluate_single_model_instance(
    model_type,
    X,
    y,
    X_test,
    k=K_FOLDS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE,
):
    """
    Esegue l'istanza di esecuzione per un singolo modello:
    1. Repeated Stratified K-Fold CV con calcolo metriche per ciascuno dei k * n_repeats fold.
    2. Calcolo statistiche aggregate (medie e deviazione standard).
    3. Calcolo metriche complessive Out-Of-Fold (OOF), mediate tra le ripetizioni.
    4. Generazione predizioni medie sul test set (media su k * n_repeats modelli).
    """
    rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=random_state)
    n_total = k * n_repeats

    oof_probs = np.zeros(len(X))
    test_probs = np.zeros(len(X_test))
    fold_metrics_list = []

    print(f"\n{'='*60}")
    print(f"Avvio {n_repeats}x Stratified {k}-Fold CV con modello: {model_type.upper()}")
    print(f"{'='*60}")

    for i, (train_idx, val_idx) in enumerate(rskf.split(X, y)):
        repeat, fold = divmod(i, k)  # i = 0..14 -> (repeat 0..2, fold 0..4)
        X_train_f, X_val_f = X.iloc[train_idx], X.iloc[val_idx]
        y_train_f, y_val_f = y.iloc[train_idx], y.iloc[val_idx]

        model = build_model(model_type, random_state=random_state)
        model.fit(X_train_f, y_train_f)

        # Predizione sul fold di validazione
        val_prob = model.predict_proba(X_val_f)[:, 1]
        val_pred = (val_prob >= 0.5).astype(int)
        oof_probs[val_idx] += val_prob / n_repeats

        metrics = compute_metrics(y_val_f, val_pred, val_prob)
        fold_metrics_list.append(metrics)

        print(f"Repeat {repeat+1}/{n_repeats} | Fold {fold+1}/{k} -> Accuracy: {metrics['accuracy']:.4f} | " f"F1: {metrics['f1']:.4f} | ROC-AUC: {metrics['roc_auc']:.4f}")

        # Accumula predizione sul test set
        test_probs += model.predict_proba(X_test)[:, 1] / n_total

    # Statistiche aggregate sui K fold
    mean_acc = np.mean([m["accuracy"] for m in fold_metrics_list])
    std_acc = np.std([m["accuracy"] for m in fold_metrics_list])
    mean_f1 = np.mean([m["f1"] for m in fold_metrics_list])
    mean_auc = np.mean([m["roc_auc"] for m in fold_metrics_list])

    # Performance Out-Of-Fold complessiva
    oof_pred = (oof_probs >= 0.5).astype(int)
    overall_oof_acc = accuracy_score(y, oof_pred)
    overall_oof_auc = roc_auc_score(y, oof_probs)

    # Stampa risultati aggregati nel formato esatto
    print(f"\n{'-'*60}")
    print(f"RISULTATI CV AGGREGATI ({n_repeats}x{k} FOLDS = {n_total} valutazioni):")
    print(f"Accuracy Media : {mean_acc:.4f} (+/- {std_acc:.4f})")
    print(f"F1 Media       : {mean_f1:.4f}")
    print(f"ROC-AUC Media  : {mean_auc:.4f}")
    print(f"OOF Accuracy   : {overall_oof_acc:.4f}")
    print(f"OOF ROC-AUC    : {overall_oof_auc:.4f}")
    print(f"{'-'*60}\n")

    test_predictions = (test_probs >= 0.5).astype(int)

    metrics_summary = {
        "Modello": model_type,
        "Accuracy Media": mean_acc,
        "Std Accuracy": std_acc,
        "F1 Media": mean_f1,
        "ROC-AUC Media": mean_auc,
        "OOF Accuracy": overall_oof_acc,
        "OOF ROC-AUC": overall_oof_auc,
    }
    return test_predictions, test_probs, metrics_summary


# ==============================================================================
# ESECUZIONE MULTI-MODELLO CON RE-INDIRIZZAMENTO SU UNICO FILE
# ==============================================================================
benchmark_results = []
all_test_predictions = {}

with TeeOutput(BENCHMARK_OUTPUT_FILE, mode="w") as tee:
    print("=" * 75)
    print("ESECUZIONE BENCHMARK DEI MODELLI DEFINITI A INIZIO NOTEBOOK")
    print(f"Data/Ora: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Modelli in esame: {MODELS_TO_BENCHMARK}")
    print(f"K-Folds: {K_FOLDS} | Repeats: {N_REPEATS} | Random State: {RANDOM_STATE}")
    print(f"Numero di Feature: {len(feature_cols)}")
    print("=" * 75)

    for model_name in MODELS_TO_BENCHMARK:
        print("\n" + "#" * 75)
        print(f"### AVVIO ISTANZA: MODELLO = '{model_name.upper()}'")
        print("#" * 75)

        # 1. Addestramento e Valutazione CV
        test_preds, test_probs, summary = evaluate_single_model_instance(
            model_type=model_name,
            X=X_train,
            y=y_train,
            X_test=X_test,
            k=K_FOLDS,
            n_repeats=N_REPEATS,
            random_state=RANDOM_STATE,
        )

        benchmark_results.append(summary)
        all_test_predictions[model_name] = test_preds

        # 2. Generazione e salvataggio file di sottomissione per il modello corrente
        model_sub_path = f"submission_{model_name}.csv"
        predict_and_save(
            test_preds,
            test_battle_ids=test_df["battle_id"],
            output_path=model_sub_path,
        )

    # 3. Costruzione della tabella riassuntiva delle metriche
    df_metrics = pd.DataFrame(benchmark_results)

    # Calcolo della media complessiva delle performance (su tutti gli indici) per ciascun modello
    indices_cols = [
        "Accuracy Media",
        "F1 Media",
        "ROC-AUC Media",
        "OOF Accuracy",
        "OOF ROC-AUC",
    ]
    df_metrics["Media Indici Modello"] = df_metrics[indices_cols].mean(axis=1)

    print("\n" + "=" * 75)
    print("TABELLA RIASSUNTIVA PERFORMANCE (MEDIE E METRICHE):")
    print("=" * 75)
    print(df_metrics.to_string(index=False))

    # Media delle performance tra tutti i modelli per ciascuna metrica
    mean_across_models = df_metrics[indices_cols + ["Media Indici Modello"]].mean()
    print("\n" + "-" * 75)
    print("MEDIA DELLE PERFORMANCE TRA TUTTI I MODELLI:")
    for metric_name, val in mean_across_models.items():
        print(f"  {metric_name:<25}: {val:.4f}")
    print("-" * 75)
    print(f"\nTutti gli output dell'esecuzione sono stati salvati con successo in: '{BENCHMARK_OUTPUT_FILE}'\n")

# Visualizzazione della tabella in formato DataFrame nel notebook
print("Tabella Riepilogativa delle Performance:")
display(df_metrics)

# ==============================================================================
# ISTOGRAMMA DI CONFRONTO TRA I VARI MODELLI PER CIASCUNA METRICA
# ==============================================================================
metrics_to_plot = [
    "Accuracy Media",
    "F1 Media",
    "ROC-AUC Media",
    "OOF Accuracy",
    "OOF ROC-AUC",
]

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
fig, ax = plt.subplots(figsize=(14, 7), dpi=120)

x = np.arange(len(metrics_to_plot))
n_models = len(df_metrics)
total_group_width = 0.8
bar_width = total_group_width / n_models

# Palette colori per distinguere chiaramente i vari modelli
colors = ["#1f77b4", "#2ca02c", "#ff7f0e", "#9467bd"]

for i, row in df_metrics.iterrows():
    model_name = row["Modello"]
    vals = [row[m] for m in metrics_to_plot]
    offset = (i - (n_models - 1) / 2) * bar_width
    bars = ax.bar(
        x + offset,
        vals,
        width=bar_width,
        label=model_name,
        color=colors[i % len(colors)],
        edgecolor="black",
        linewidth=0.7,
        alpha=0.9,
    )

    # Etichetta con valore numerico a 4 decimali sopra ciascuna barra
    for bar in bars:
        height = bar.get_height()
        ax.annotate(
            f"{height:.4f}",
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8.5,
            fontweight="bold",
        )

ax.set_title("Confronto Performance dei Modelli per Ciascuna Metrica", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("Metrica di Valutazione", fontsize=12, fontweight="bold", labelpad=10)
ax.set_ylabel("Punteggio (Score)", fontsize=12, fontweight="bold", labelpad=10)
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, fontsize=11, fontweight="bold")

# Margine dinamico sull'asse Y per esaltare le differenze mantenendo coerenza visiva
min_val = min(df_metrics[metrics_to_plot].min())
ax.set_ylim(max(0, min_val - 0.08), 1.05)

ax.legend(title="Modello", title_fontsize="11", fontsize=10, loc="lower right", frameon=True, shadow=True)
ax.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.savefig(HISTOGRAM_OUTPUT_FILE, dpi=300, bbox_inches="tight")
plt.show()
print(f"Grafico istogramma salvato con successo in '{HISTOGRAM_OUTPUT_FILE}'.")